# csvdelta: row- and cell-level diffs between two CSV files

This notebook is a short tour of the public API of `csvdelta`. We will:

1. write two small CSV files,
2. diff them with a key column,
3. look at row and cell level changes,
4. use a numeric tolerance,
5. diff tables that were loaded with `read_table`,
6. save the delta as JSON.

Install the package first with `pip install csvdelta` (Python 3.9 or newer).

In [ ]:
import csvdelta

print("csvdelta version:", csvdelta.__version__)

## 1. Create example files

We write two small files, `old.csv` and `new.csv`. Row `1` is unchanged, row `2` has a new amount, row `3` is gone from the new file, and row `4` is new.

In [ ]:
import csv

old_rows = [
    ["id", "name", "amount"],
    [1, "Ann", 10.0],
    [2, "Bob", 20.0],
    [3, "Cho", 30.0],
]
new_rows = [
    ["id", "name", "amount"],
    [1, "Ann", 10.0],
    [2, "Bob", 25.0],
    [4, "Dan", 40.0],
]

for path, rows in [("old.csv", old_rows), ("new.csv", new_rows)]:
    with open(path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)

print(open("old.csv", encoding="utf-8").read())
print(open("new.csv", encoding="utf-8").read())

## 2. Diff two files by key

`diff_files` reads both paths and returns a `DiffResult`. Passing `key="id"` tells the algorithm to match rows by the `id` column instead of by position. `summarize` turns the result into a plain dictionary of counts.

In [ ]:
result = csvdelta.diff_files("old.csv", "new.csv", key="id")

print(csvdelta.summarize(result))

## 3. Inspect row and cell changes

`result.row_changes` is a list of `RowChange` objects. Each one has a `kind` (`added`, `removed` or `changed`), a `key`, and a list of `cells`. For changed rows, every entry in `cells` is a `CellChange` with `column`, `old` and `new`.

In [ ]:
for change in result.row_changes:
    print(f"{change.kind:<8} key={change.key}")
    for cell in change.cells or []:
        print(f"           {cell.column}: {cell.old} -> {cell.new}")

## 4. Numeric tolerance

Numeric cells are equal when `abs(a - b) <= tolerance`. The default tolerance is `0.0`, so the change from `20.0` to `25.0` is reported. With a tolerance of `10.0` that change is treated as equal, so only the added and removed rows remain.

In [ ]:
loose = csvdelta.diff_files("old.csv", "new.csv", key="id", tolerance=10.0)

print(csvdelta.summarize(loose))

## 5. Diff tables loaded in memory

If you already have the data in memory, or you want to inspect it before diffing, load each file with `read_table`. It returns a `Table` with a `header` list and a `rows` list. Then pass the two tables to `diff_tables`. The result is the same as with `diff_files`.

In [ ]:
table_a = csvdelta.read_table("old.csv")
table_b = csvdelta.read_table("new.csv")

print("header:", table_a.header)
print("rows in old:", len(table_a.rows))
print("rows in new:", len(table_b.rows))

result_tables = csvdelta.diff_tables(table_a, table_b, key="id")
print(csvdelta.summarize(result_tables))

## 6. Without a key

If no key is given, `csvdelta` aligns the rows by position using a longest common subsequence on row fingerprints. This works well when the two files have the same rows in the same order and only a few lines were inserted or deleted.

In [ ]:
positional = csvdelta.diff_files("old.csv", "new.csv")

print(csvdelta.summarize(positional))

## 7. Save the delta as JSON

`write_delta` writes a `DiffResult` to disk. The `json` format uses version `1` of the delta schema, with the keys `added`, `removed` and `changed`. The format can also be `text` or `html`.

In [ ]:
csvdelta.write_delta(result, "delta.json", fmt="json")

with open("delta.json", encoding="utf-8") as f:
    print(f.read())

## 8. The same diff from the command line

The `csvdelta` console script wraps the same functions. The text output marks added rows with `+`, removed rows with `-` and changed cells with `~`. The exit code is `0` when the files are identical, `1` when there are differences and `2` on error.

In [ ]:
!csvdelta diff old.csv new.csv --key id

In [ ]:
import os

for path in ["old.csv", "new.csv", "delta.json"]:
    if os.path.exists(path):
        os.remove(path)

print("demo files removed")